MERGED ANALYSIS EDA

In [2]:
import pandas as pd

In [3]:
df1 = pd.read_csv("/home/melody/carjacking-detection/data/parquet_cleaned.csv")
 

In [5]:
df2 = pd.read_csv("/home/melody/carjacking-detection/data/Vehicle_Data.csv")

RUNNING THE MERGE FIRST

In [11]:
df2['IMEI'] = df2['IMEI'].astype(int)
df1['sourceimei'] = df1['sourceimei'].astype(int)

df_merged = df1.merge(df2, left_on='sourceimei', right_on='IMEI', how='left')

print(df_merged.shape)
 
print(df_merged.isnull().sum())

(58374, 45)
sourceimei                           0
battery_voltage_value                0
hardware_attached_gps_not_present    0
unitdatetime                         0
latitude                             0
longitude                            0
altitude                             0
speed                                0
heading                              0
bearing                              0
odometer                             0
gforce_forward                       0
gforce_backward                      0
gforce_up                            0
gforce_down                          0
gforce_left                          0
gforce_right                         0
engine_hours                         0
classification                       0
condition                            0
country                              0
distance_suburb                      0
municipality                         0
postal_code                          0
province                             0
road         

FEATURE ENGINEERING

In [12]:
#PARSE DATETIME
df_merged['unitdatetime'] = pd.to_datetime(df_merged['unitdatetime'])

In [13]:
#time features
df_merged['hour'] = df_merged['unitdatetime'].dt.hour
df_merged['dayofweek'] = df_merged['unitdatetime'].dt.dayofweek  # 0=Monday, 6=Sunday
df_merged['month'] = df_merged['unitdatetime'].dt.month
df_merged['is_weekend'] = df_merged['dayofweek'].isin([5, 6]).astype(int)
df_merged['is_night'] = df_merged['hour'].between(20, 23) | df_merged['hour'].between(0, 3)
df_merged['is_night'] = df_merged['is_night'].astype(int)

In [16]:
# vehicle age
df_merged['vehicle_age'] = 2026 - df_merged['Year']

# gforce total magnitude
df_merged['gforce_magnitude'] = (
    df_merged['gforce_forward'] +
    df_merged['gforce_backward'] +
    df_merged['gforce_left'] +
    df_merged['gforce_right']
)

In [15]:
# risk flags
df_merged['is_stationary'] = (df_merged['speed'] == 0).astype(int)
df_merged['gforce_anomaly'] = (
    (df_merged['gforce_forward'] > 200) |
    (df_merged['gforce_left'] > 200) |
    (df_merged['gforce_right'] > 200) |
    (df_merged['gforce_backward'] > 200)
).astype(int)

df_merged['risk_event'] = (
    (df_merged['is_stationary'] == 1) & (df_merged['is_night'] == 1) |
    (df_merged['gforce_anomaly'] == 1)
).astype(int)

# encode gender
df_merged['gender_encoded'] = (df_merged['Gender'] == 'Male').astype(int)

print(df_merged[['hour', 'is_night', 'is_weekend', 'vehicle_age', 
                  'gforce_magnitude', 'risk_event']].head(10))
print(f"\nTotal risk events: {df_merged['risk_event'].sum()}")

   hour  is_night  is_weekend  vehicle_age  gforce_magnitude  risk_event
0    23         1           0          5.0                50           1
1    15         0           0          5.0                50           0
2    15         0           0          5.0                50           0
3    15         0           0          5.0                50           0
4    15         0           0          5.0                50           0
5    15         0           0          5.0                50           0
6    15         0           0          5.0                50           0
7    15         0           0          5.0                 0           0
8    15         0           0          5.0                 0           0
9    15         0           0          5.0                50           0

Total risk events: 862


In [17]:
df_merged.to_csv('/home/melody/carjacking-detection/data/merged_cleaned.csv', index=False)
print("saved")

saved
